In [27]:
from typing import TypedDict, Annotated
from langgraph.graph import END, StateGraph, add_messages
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage , AIMessage, ToolMessage
from langchain_community.tools import TavilySearchResults
from langchain_core.tools import tool
from dotenv import load_dotenv
from langgraph.prebuilt import ToolNode
from pydantic import BaseModel, Field
from typing import Annotated,List , Optional, Dict, Any
from langchain_core.prompts import PromptTemplate

load_dotenv()

True

In [34]:
llm = ChatGoogleGenerativeAI(model='gemini-2.0-flash')
search_tool = TavilySearchResults(max_results=5, search_depth="advanced")


E0000 00:00:1758565445.657851  723368 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.


In [29]:
class PlanningAgentOutput(BaseModel):
  """ Ouput Schema for The Planning Agent """
  domain : str  =  Field(..., description="The domain That The user query Belongs to")
  requires_prereqs : bool =  Field(..., description="Whether or not The Given topic Requires a prerequisites")
  reasoning :str = Field(..., description="Reasoning String")
  search_queries :list[str] =  Field(..., description="List of search queries")

class Lesson(BaseModel):
    title: str = Field(..., description = "Title of the lesson")
    description: str = Field(..., description = "Description of the lesson")
    estimated_time_hours: Optional[float] = None

class Module(BaseModel):
    title: str = Field(..., description = "Title of the module")
    description: str = Field(..., description = "Description of the module")
    lessons: List[Lesson] = Field(..., description = "List of lessons in the module")

class RoadmapPlannerOutput(BaseModel):
    modules: List[Module] = Field(..., description = "List of modules in the roadmap")

In [36]:
class RoadmapState(TypedDict):
    messages: Annotated[list, add_messages]
    domain: Optional[str]
    requires_prereqs: Optional[bool]
    reasoning: Optional[str]
    search_queries: Optional[List[str]]
    search_results: Optional[List[Dict[str, Any]]]
    modules: Optional[List[Module]]

In [30]:
Planner_Agent_Prompt = """ You are an expert curriculum designer and search strategist.
Task:
Given a user topic, (1) classify its domain, (2) decide whether prerequisites are required, (3) explain your decision briefly, and (4) produce a set of robust web search queries that will retrieve high-quality material covering the full topic from intro → internals → advanced, including prerequisite checks, syllabi, tutorials, projects, and reference material.

Requirements / Rules:
1. Analyze the user's topic carefully and classify it into a clear, broad domain (e.g., "Cloud Computing", "Networking", "Artificial Intelligence", "Mathematics", "Cybersecurity", "Software Engineering").
2. Decide whether the topic **requires prerequisites** (true/false). "Requires prerequisites" means the learner will likely need prior foundational knowledge to learn this topic effectively.
3. Provide a short human-readable **reasoning** (1–2 sentences) for the classification and prereq decision — useful for debugging.
4. Generate **6 to 8 robust search queries** (strings). Each query should be crafted to retrieve comprehensive resources for learning the topic end-to-end:
   - include queries that fetch **overview/introduction**, **prerequisite lists**, **detailed internals/architecture**, **course syllabi**, **practical tutorials / hands-on labs**, **project ideas**, and **reference docs / cheatsheets**.
   - include a few targeted queries (e.g., `site:edu`, `site:github.com`, `filetype:pdf`) where appropriate to increase coverage of authoritative sources.
   - prefer explicit intent phrases: "how to learn", "roadmap", "syllabus", "tutorial", "course", "projects", "best practices", "implementation", "lab".
   - make queries specific (e.g., include common subtopics or acronyms if relevant).
5. Output MUST be **ONLY** a single valid JSON object conforming exactly to the schema below — no extra text, no explanation, no code fences.

Input:
Topic: {user_query}

Output JSON Schema:
{{
  "domain": "<string>",
  "requires_prereqs": <true|false>,
  "reasoning": "<short explanation (1-2 sentences)>",
  "search_queries": ["<query1>", "<query2>", ...]  // 6-8 items
}}

Example:
Input Topic: "Cloud Security"
(Valid JSON output example:)
{{
  "domain": "Cloud Computing / Cybersecurity",
  "requires_prereqs": true,
  "reasoning": "Cloud security builds on networking, virtualization, and IAM concepts; learners should know networking basics and cloud models.",
  "search_queries": [
    "Cloud security roadmap how to learn from basics to advanced",
    "prerequisites for cloud security networking virtualization IAM",
    "cloud security syllabus site:edu filetype:pdf",
    "cloud security hands-on labs tutorials site:github.com",
    "IAM best practices AWS Azure GCP tutorial",
    "cloud network security architecture introduction"
  ]
}}
 """

RoadMap_Agent_Prompt = """You are an expert curriculum planner and learning path designer.

Your task is to generate a structured learning roadmap for a given topic,
based on domain classification, prerequisite needs, and relevant search results.

### Input Context
Domain: {domain}
Requires prerequisites: {requires_prereqs}
Reasoning: {reasoning}

Search results:
{search_results}

### Instructions
1. Analyze the search results to extract the main topics, subtopics, and logical learning order.
2. Organize the content into a sequence of **modules**. Each module should cover a coherent set of topics.
3. Break each module into **lessons**:
   - Include a clear title.
   - Provide a short description (1–2 sentences).
   - Optionally, estimate the learning time in hours (roughly).
4. Respect prerequisites:
   - If requires_prereqs is true, create an **Introductory Module** that covers the necessary background.
   - Otherwise, start directly with the main topic.
5. The roadmap should move from **fundamentals → core concepts → advanced concepts → applications/projects**.
6. Output ONLY a valid JSON object that fits this schema:

{{{{
  "modules": [
    {{{{
      "title": "<module title>",
      "description": "<module description>",
      "lessons": [
        {{{{
          "title": "<lesson title>",
          "description": "<short lesson description>",
          "estimated_time_hours": <float or null>
        }}}}
      ]
    }}}}
  ]
}}}}

### Example
Input Topic: "Cloud Security"
Output:
{{{{
  "modules": [
    {{{{
      "title": "Prerequisite Knowledge",
      "description": "Essential background in networking, cloud fundamentals, and identity access management.",
      "lessons": [
        {{{{"title": "Networking Basics", "description": "Covers TCP/IP, routing, and firewalls.", "estimated_time_hours": 4}}}},
        {{{{"title": "Cloud Service Models", "description": "Understanding IaaS, PaaS, SaaS models.", "estimated_time_hours": 3}}}},
        {{{{"title": "Identity & Access Management", "description": "Intro to authentication, authorization, and IAM tools.", "estimated_time_hours": 3}}}}
      ]
    }}}},
    {{{{
      "title": "Core Cloud Security Concepts",
      "description": "Learn the fundamental practices for securing cloud environments.",
      "lessons": [
        {{{{"title": "Shared Responsibility Model", "description": "Explains security roles of providers and users.", "estimated_time_hours": 2}}}},
        {{{{"title": "Cloud Encryption Techniques", "description": "Covers data-at-rest and in-transit encryption.", "estimated_time_hours": 3}}}}
      ]
    }}}}
  ]
}}}}
"""

planner_prompt_template = PromptTemplate(
    input_variables=["user_query"],
    template=Planner_Agent_Prompt,
)

roadmap_agent_template = PromptTemplate(
    input_variables=["domain", "requires_prereqs", "reasoning", "search_results"],
    template=RoadMap_Agent_Prompt,
)

In [8]:
llm = ChatGoogleGenerativeAI(model='gemini-2.0-flash').with_structured_output(PlanningAgentOutput)

planner_chain = planner_prompt_template | llm
response = planner_chain.invoke({"user_query": "Cloud Security"})
response

E0000 00:00:1758563427.928558  723368 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.


PlanningAgentOutput(domain='Cloud Computing / Cybersecurity', requires_prereqs=True, reasoning='Cloud security builds upon foundational knowledge of networking, cloud computing, and security principles.  A basic understanding of these areas is necessary for effective learning.', search_queries=['"Cloud security roadmap" how to learn from basics to advanced', '"prerequisites for cloud security" networking virtualization IAM', '"cloud security syllabus" site:edu filetype:pdf', '"cloud security hands-on labs tutorials" site:github.com', 'IAM best practices AWS Azure GCP tutorial', '"cloud network security architecture" introduction', '"cloud security best practices" implementation', 'AWS Azure GCP cloud security projects'])

In [31]:
@tool
def search_queries_tool(search_queries: List[str]) -> List[Dict[str, Any]]:
    """
    Execute multiple search queries and return structured results.
    
    Args:
        search_queries: List of search query strings
        
    Returns:
        List of dictionaries containing search results for each query
    """
    all_results = []
    
    for i, query in enumerate(search_queries):
        try:
            search_results = search_tool.invoke({"query": query})
            
            query_results = {
                "query": query,
                "query_index": i + 1,
                "results": []
            }
            for result in search_results:
                structured_result = {
                    "title": result.get("title", ""),
                    "url": result.get("url", ""),
                    "content": result.get("content", "")[:500],
                    "score": result.get("score", 0)
                }
                query_results["results"].append(structured_result)
            
            query_results["total_results"] = len(query_results["results"])
            all_results.append(query_results)
            
        except Exception as e:
            error_result = {
                "query": query,
                "query_index": i + 1,
                "error": str(e),
                "results": [],
                "total_results": 0
            }
            all_results.append(error_result)
    
    return all_results

search_tool_node = ToolNode([search_queries_tool])

# def extract_search_queries_from_response(planning_response: PlanningAgentOutput) -> List[str]:
#     """Extract search queries from the planning agent response."""
#     return planning_response.search_queries

# def execute_search_pipeline(planning_response: PlanningAgentOutput) -> List[Dict[str, Any]]:
#     """
#     Execute search pipeline using planning agent output.
    
#     Args:
#         planning_response: Output from the planning agent
        
#     Returns:
#         Structured search results for all queries
#     """
#     search_queries = extract_search_queries_from_response(planning_response)
#     search_results = search_queries_tool.invoke({"search_queries": search_queries})
#     return search_results

In [37]:
def planning_node(state: RoadmapState) -> RoadmapState:
    """
    Planning node that analyzes user query and generates search queries.
    """
    planning_response = planner_chain.invoke({"user_query": state["messages"][-1].content})
    state["domain"] = planning_response.domain
    state["requires_prereqs"] = planning_response.requires_prereqs
    state["reasoning"] = planning_response.reasoning
    state["search_queries"] = planning_response.search_queries
    return state


def search_node(state: RoadmapState) -> RoadmapState:
    """
    Search node that executes search queries and returns results.
    """
    if not state.get("search_queries"):
        raise ValueError("No search queries found in state")
    
    search_results = search_queries_tool.invoke({"search_queries": state["search_queries"]})
    
    state["search_results"] = search_results
    
    total_results = sum(len(query_result.get("results", [])) for query_result in search_results)
    tool_message = ToolMessage(
        content=f"Search completed. Found {total_results} total results across {len(search_results)} queries.",
        tool_call_id="search_queries_tool"
    )
    state["messages"].append(tool_message)
    
    return state

def roadmap_node(state: RoadmapState) -> RoadmapState:
    """
    Roadmap node that generates the final learning roadmap.
    """
    if not all(key in state for key in ["domain", "requires_prereqs", "reasoning", "search_results"]):
        raise ValueError("Missing required state information for roadmap generation")
    
    search_results_text = ""
    for query_result in state["search_results"]:
        search_results_text += f"\nQuery: {query_result['query']}\n"
        for result in query_result.get("results", [])[:3]:  # Top 3 results per query
            search_results_text += f"- {result['title']}: {result['content'][:200]}...\n"
    
    # Execute roadmap generation
    roadmap_llm = ChatGoogleGenerativeAI(model='gemini-2.0-flash').with_structured_output(RoadmapPlannerOutput)
    roadmap_chain = roadmap_agent_template | roadmap_llm
    
    roadmap_result = roadmap_chain.invoke({
        "domain": state["domain"],
        "requires_prereqs": state["requires_prereqs"],
        "reasoning": state["reasoning"],
        "search_results": search_results_text
    })
    
    return {"modules" : roadmap_result}

# Create the workflow graph
graphBuilder = StateGraph(RoadmapState)

graphBuilder.add_node("planning", planning_node)
graphBuilder.add_node("search", search_node)
graphBuilder.add_node("roadmap", roadmap_node)

graphBuilder.set_entry_point("planning")
graphBuilder.add_edge("planning", "search")
graphBuilder.add_edge("search", "roadmap")
graphBuilder.add_edge("roadmap", END)

app = graphBuilder.compile()

print(app.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	planning(planning)
	search(search)
	roadmap(roadmap)
	__end__([<p>__end__</p>]):::last
	__start__ --> planning;
	planning --> search;
	search --> roadmap;
	roadmap --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [38]:
result = app.invoke({"messages" : [HumanMessage(content="I want to learn Cloud Security from scratch.") ]})
result

E0000 00:00:1758565534.733288  723368 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.


{'messages': [HumanMessage(content='I want to learn Cloud Security from scratch.', additional_kwargs={}, response_metadata={}, id='56f8132f-e6b6-40da-bba5-efd0c3422a27'),
  ToolMessage(content='Search completed. Found 40 total results across 8 queries.', id='8eb4d140-66bf-4325-a7bd-50ad50f6fe29', tool_call_id='search_queries_tool')],
 'domain': 'Cloud Computing / Cybersecurity',
 'requires_prereqs': True,
 'reasoning': 'Cloud security builds upon foundational knowledge of networking, cloud computing, and security principles.  A basic understanding of these areas is necessary for effective learning.',
 'search_queries': ['"cloud security" roadmap how to learn from basics to advanced',
  '"cloud security" prerequisites networking virtualization IAM',
  '"cloud security" syllabus site:edu filetype:pdf',
  '"cloud security" hands-on labs tutorials site:github.com',
  'IAM best practices AWS Azure GCP tutorial',
  'cloud network security architecture introduction',
  'cloud security best pr